In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Literal, Optional

import numpy as np
from numpy.typing import NDArray
from PIL import Image

# Make an alias for the image format type to save space
ImageF = NDArray[np.float32]  # HxWxC in [0,1]

In [3]:
def load_image_rgb(path: str | Path) -> ImageF:
    """Load an image in RGB from a specified file path."""
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def save_image_rgb(path: str | Path, img: ImageF) -> None:
    """Save an image in RGB to a specified file path."""
    img8 = np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)
    Image.fromarray(img8, mode="RGB").save(path)

## Pipeline Algorithms
---
### *Algorithm 1: RGB-to-Luma Conversion*  
Compute a weighted sum of the red, green, and blue channels to produce a single-channel image representing perceived brightness, i.e., **luma** ($Y'$). The weights are derived from the ITU-R BT.709 standard, which models human photopic vision sensitivity: $0.2126$ for red, $0.7152$ for green, and $0.0722$ for blue.

Note that this operation is a weighted *channel reduction*, which irreversibly destroys chromatic information. Ergo, reserve this operation for grayscale pipelines.

**Input:** $I \in \mathbb{R}^{H\times W\times 3}$  
**Output:** $Y' \in \mathbb{R}^{H\times W}$

1. $\mathbf{for}\;i \leftarrow \mathbf{to}\; H\; \mathbf{do}$
2. $\quad \mathbf{for}\; j \leftarrow \mathbf{to}\; W\; \mathbf{do}$
3. $\quad\quad R \leftarrow I_{i,j,0}$
4. $\quad\quad G \leftarrow I_{i,j,1}$
5. $\quad\quad B \leftarrow I_{i,j,2}$
6. $\quad\quad Y'_{i,j} \leftarrow 0.2126R + 0.7152G + 0.0722B$
7. $\mathbf{return}\; Y'$
---

### *Algorithm 2: Luma-to-RGB*  
Convert a single-channel luma image into a 3-channel grayscale RGB image by replicating the luma values across all three color channels (R, G, B). This function primarily exists to make a grayscale image displayable in systems expecting RGB.

Note that this operation is not the inverse of $\mathtt{rgb\_to\_luma(rgb\_img)}$; chromatic information is permanently lost once it has been discarded.

**Input:** $Y' \in \mathbb{R}^{H\times W}$  
**Output:** $I \in \mathbb{R}^{H\times W\times3}$

1. $\mathbf{for}\;i\leftarrow1\;\mathbf{to}\;H\;\mathbf{do}$
2. $\quad \mathbf{for}\;j\leftarrow1\;\mathbf{to}\;W\;\mathbf{do}$
3. $\quad\quad I_{i,j,0}\leftarrow Y_{i,j}$
4. $\quad\quad I_{i,j,1}\leftarrow Y_{i,j}$
5. $\quad\quad I_{i,j,2}\leftarrow Y_{i,j}$
6. $\mathbf{return}\;I$
---

### *Algorithm 3: sRGB-to-Linear Decoding*  

Convert gamma-encoded sRGB values into linear-light values using the sRGB decoding function. Image arithmetic (blending, filtering, convolution) should occur in linear space since sRGB encoding is optimized for perceptual uniformity and storage efficiency, not physical accuracy.

**Input:** $X \in \mathbb{R}^{H\times W\times C}$  
**Output:** $L \in \mathbb{R}^{H\times W\times C}$

1. $a\leftarrow0.055$
2. $\mathbf{for}\;\text{each pixel}\;(i,j)\;\text{and channel}\;\mathbf{do}$
3. $\quad x\leftarrow X_{i,j,c}$
4. $\quad \mathbf{if}\;x\leq0.04045\;\mathbf{then}$
5. $\quad\quad L_{i,j,c}\leftarrow x/12.92$
6. $\quad \mathbf{else}$
7. $\quad\quad L_{i,j,c}\leftarrow\frac{x+a}{1+a}^{2.4}$
8. $\mathbf{return}$
---

In [6]:
def rgb_to_luma(rgb_img: ImageF) -> ImageF:
    """Conform an RGB image to Rec.709's luma coefficients."""
    r, g, b = rgb_img[..., 0], rgb_img[..., 1], rgb_img[..., 2]
    luma_img = 0.2126*r + 0.7152*g + 0.0722*b
    return luma_img.astype(np.float32)

def luma_to_grayrgb(luma_img: ImageF) -> ImageF:
    return np.stack([luma_img, luma_img, luma_img], axis=-1).astype(np.float32)

def srgb_to_linear(srgb: ImageF) -> ImageF:
    """Piecewise implementation of the sRGB inverse EOTF."""
    a = 0.055
    return np.where(
        srgb <= 0.04045,
        srgb / 12.92,
        ((srgb + a) / (1.0 + a)) ** 2.4,
    ).astype(np.float32)